# Walkthrough: show me where it says that

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fuellabs-ai/hobbylm-embed-cookbook/blob/main/show-me-where/walkthrough.ipynb)

This notebook runs the show-me-where recipe end to end. Two cut-offs give
three verdicts: "Found" returns the section that backs an answer with its
file and heading, "Not in these documents" shows the nearest sections and
feeds a gaps list, and "Check this" between the cut-offs asks a person to
confirm. It ends by calibrating the cut-offs on labelled questions.

Embeddings only. No LLM writes an answer.

In [ ]:
# One setup cell. Edit REPO, BRANCH and FOLDER here if you forked the cookbook.

REPO = "fuellabs-ai/hobbylm-embed-cookbook"
BRANCH = "main"
FOLDER = "show-me-where"

import os
from pathlib import Path

if Path("show_me.py").exists():
    print("Using the local copy of the cookbook in", os.getcwd())
else:
    if not Path(FOLDER).exists():
        get_ipython().system("git clone --depth 1 -b {BRANCH} https://github.com/{REPO}.git {FOLDER}")
    os.chdir(FOLDER)
    print("Using a fresh clone in", os.getcwd())
    print("No GPU needed; the embedding model is 140M parameters.")

In [ ]:
# Install dependencies. On Colab sentence-transformers is already there;
# pypdf and python-docx are needed for PDF and DOCX files.

%pip install -q "sentence-transformers>=5" pypdf python-docx

In [ ]:
# Ask one question the documents do answer. The reply names the file and
# the heading, so a human can check it in one click.

!python show_me.py examples/support_refund_policy "How long does a refund take to reach my card?"

In [ ]:
# Ask one question the documents do not answer. The verdict is
# "Not in these documents" and the three nearest sections are labelled
# "closest, but not an answer". The question is added to gaps.md.

!python show_me.py examples/support_refund_policy "Do you offer gift cards that never expire?"

In [ ]:
# Ask the borderline question. Its score lands between the two
# cut-offs, so the verdict is "Check this": the top section is shown
# and a person should confirm it. The question goes to check_this.md.

!python show_me.py examples/support_refund_policy "can I get a refund after 30 days?"

In [ ]:
# Batch mode: read questions.csv, write results.csv, gaps.md and
# check_this.md.
# questions.csv rows look like:
#     question,label
#     "Can I get a refund if my flowers arrive wilted?",answerable
# A label is optional; unlabelled rows still get a verdict.

!python show_me.py examples/support_refund_policy --questions examples/support_refund_policy/questions.csv --out-dir output

In [ ]:
# The gaps list groups unanswered questions by the section they scored
# highest against. Several questions under one section is a hint for an
# article that does not exist yet.

print(open("output/gaps.md").read())

In [ ]:
# The Check this list holds the questions between the cut-offs. A
# person should confirm each one before anyone repeats it as an
# answer.

print(open("output/check_this.md").read())

In [ ]:
# Calibrate the two cut-offs on the labelled questions. The upper cut-off
# is the smallest one where wrong answers reach their floor; the lower
# cut-off is the largest one where wrong refusals reach their floor. The
# script reports how many questions land in Check this and saves both
# cut-offs to cutoff.json inside the example folder, which show_me.py
# picks up from there.

!python calibrate.py examples/support_refund_policy

## Limits

The "not in these documents" decision is a score cut-off, not understanding. It will sometimes say no when the answer is there, and sometimes show a close-sounding section that does not answer the question, so set the cut-off on your own questions and keep a person in the loop for anything involving money, law or health.

A fixed cut-off does not carry over between models or document sets. Set
the pair from 30 to 50 of your own labelled questions with `calibrate.py`.

The Check this band narrows where the limit bites: borderline scores go to
a person instead of becoming a yes or a no. The two cut-offs are still
score cut-offs, so the limit does not go away.

The recipe is English only. It finds sections; it does not write answers.

## Where to go next

- `README.md` in this folder has the full quickstart and the calibration
  tables for all five example folders.
- `examples/` holds five fictional document sets, each with 10 answerable
  and 10 unanswerable questions.
- The sample companies and policies are fictional. They exist to test the
  recipe, not to describe a real product.